# 🎯 Projeto OES — Módulos da Pessoa A

**Responsável:** A  
**Referência:** Demšar (2006) — Statistical Comparisons of Classifiers over Multiple Data Sets

---

## 📋 Atividades cobertas neste notebook

| Cód | Atividade | Entregável |
|-----|-----------|------------|
| T5  | Lógica de Seleção Dinâmica (DES) | `DynamicEnsembleSelector` (k-NN local) |
| T6  | Engine de Extração de Métricas | `evaluate_all()` — sMAPE, MRE, MASE, NSE, COD |
| T7  | Validação Estatística de Demšar | Friedman + Bonferroni-Dunn + CD Diagram |

---

### ⚡ Contrato de Interface com M
- **Gabarito (Matriz de Previsões):** `np.ndarray` de shape `(N_instancias_treino, M_modelos)`
- **Pool de teste:** `np.ndarray` de shape `(N_instancias_teste, M_modelos)`
- Enquanto M não entregar os dados reais, os mocks abaixo simulam o ambiente completo.

## 0️⃣ Instalação de dependências

In [ ]:
# Todas as libs já vêm no Colab, mas garantimos as versões aqui
!pip install numpy scipy matplotlib --quiet

import numpy as np
from scipy import stats
from scipy.stats import friedmanchisquare
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ Dependências OK')

---
## T6 — `evaluator.py` · Engine de Extração de Métricas

Calcula as métricas de avaliação para qualquer conjunto de predições.  
**Métricas:** sMAPE · MRE · MASE · NSE · COD (R²)

In [ ]:
# ============================================================
#  T6 - evaluator.py
#  Responsável: A
# ============================================================

def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Symmetric Mean Absolute Percentage Error (sMAPE).
    Retorna valor em %. Quanto menor, melhor.
    Intervalo: [0%, 200%]
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    numerator   = 2.0 * np.abs(y_true - y_pred)
    denominator = np.abs(y_true) + np.abs(y_pred)
    mask   = denominator != 0
    result = np.mean(numerator[mask] / denominator[mask]) * 100.0
    return round(result, 6)


def mre(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Mean Relative Error (MRE). Quanto menor, melhor.
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask   = y_true != 0
    result = np.mean(np.abs(y_true[mask] - y_pred[mask]) / np.abs(y_true[mask]))
    return round(result, 6)


def mase(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Mean Absolute Scaled Error (MASE).
    Se > 1: modelo pior que naive. Quanto menor, melhor.
    """
    y_true      = np.array(y_true, dtype=float)
    y_pred      = np.array(y_pred, dtype=float)
    numerator   = np.mean(np.abs(y_true - y_pred))
    denominator = np.mean(np.abs(y_true[1:] - y_true[:-1]))
    if denominator == 0:
        return float('inf')
    return round(numerator / denominator, 6)


def nse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Nash-Sutcliffe Efficiency (NSE).
    Intervalo: (-inf, 1]. Quanto mais próximo de 1, melhor.
    """
    y_true  = np.array(y_true, dtype=float)
    y_pred  = np.array(y_pred, dtype=float)
    ss_res  = np.sum((y_true - y_pred) ** 2)
    ss_tot  = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0:
        return float('nan')
    return round(1 - ss_res / ss_tot, 6)


def cod(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Coefficient of Determination (COD / R²). Mesma fórmula que NSE."""
    return nse(y_true, y_pred)


def evaluate_all(y_true: np.ndarray, y_pred: np.ndarray, model_name: str = 'Model') -> dict:
    """
    Calcula todas as métricas de uma vez.
    Retorna dicionário com resultados e imprime tabela formatada.
    """
    metrics = {
        'model': model_name,
        'sMAPE': smape(y_true, y_pred),
        'MRE'  : mre(y_true, y_pred),
        'MASE' : mase(y_true, y_pred),
        'NSE'  : nse(y_true, y_pred),
        'COD'  : cod(y_true, y_pred),
    }
    print(f"\n{'='*50}")
    print(f"  Métricas — {model_name}")
    print(f"{'='*50}")
    for k, v in metrics.items():
        if k != 'model':
            print(f'  {k:<8}: {v}')
    print(f"{'='*50}")
    return metrics

print('✅ T6 — evaluator carregado')

In [ ]:
# ── Teste rápido do T6 com mock data ──────────────────────────
np.random.seed(42)
y_true_mock = np.random.uniform(500, 60000, 100)
y_pred_mock = y_true_mock * np.random.uniform(0.85, 1.15, 100)  # ~15% erro

evaluate_all(y_true_mock, y_pred_mock, model_name='Mock Model')
print('\n[OK] T6 funcionando corretamente.')

---
## T5 — `des_dynamic.py` · Seleção Dinâmica de Ensemble (DES)

Para cada instância de teste, encontra os **k vizinhos mais próximos** no conjunto de treino  
e seleciona os modelos com **menor erro local (MAE)** na região de competência.

> **Contrato com M:** recebe `gabarito` como `ndarray(N_treino, M_modelos)`.

In [ ]:
# ============================================================
#  T5 - des_dynamic.py
#  Responsável: A
# ============================================================

class DynamicEnsembleSelector:
    """
    Seleção Dinâmica de Ensemble baseada em k-NN local.

    Parâmetros
    ----------
    k         : int   — número de vizinhos para a região de competência.
    threshold : float — fração dos melhores modelos a selecionar (ex: 0.3 = top 30%).
    """

    def __init__(self, k: int = 7, threshold: float = 0.3):
        self.k         = k
        self.threshold = threshold
        self.X_train   = None   # (N, F)
        self.gabarito  = None   # (N, M) — previsões de cada modelo no treino
        self.y_train   = None   # (N,)
        self.n_models  = None

    # ----------------------------------------------------------
    # 1. FIT — memoriza X_train, gabarito e y_train
    # ----------------------------------------------------------
    def fit(self, X_train: np.ndarray, gabarito: np.ndarray, y_train: np.ndarray):
        """
        X_train  : (N_treino, N_features)
        gabarito : (N_treino, M_modelos)  ← contrato de interface com M
        y_train  : (N_treino,)
        """
        self.X_train  = np.array(X_train,  dtype=float)
        self.gabarito = np.array(gabarito, dtype=float)
        self.y_train  = np.array(y_train,  dtype=float)
        self.n_models = gabarito.shape[1]
        print(f'[DES] fit() | {self.X_train.shape[0]} instâncias treino | {self.n_models} modelos no pool')
        return self

    # ----------------------------------------------------------
    # 2. DISTÂNCIA EUCLIDIANA — k vizinhos mais próximos
    # ----------------------------------------------------------
    def _get_neighbors(self, x_query: np.ndarray) -> np.ndarray:
        """Retorna índices dos k vizinhos mais próximos de x_query."""
        diffs = self.X_train - x_query          # broadcasting (N, F)
        dists = np.sqrt(np.sum(diffs ** 2, axis=1))
        return np.argsort(dists)[:self.k]

    # ----------------------------------------------------------
    # 3. COMPETÊNCIA LOCAL — MAE de cada modelo na vizinhança
    # ----------------------------------------------------------
    def _local_competence(self, neighbor_idx: np.ndarray) -> np.ndarray:
        """Retorna array (M_modelos,) com o MAE local de cada modelo."""
        y_local     = self.y_train[neighbor_idx]         # (k,)
        preds_local = self.gabarito[neighbor_idx, :]     # (k, M)
        return np.mean(np.abs(preds_local - y_local[:, np.newaxis]), axis=0)  # (M,)

    # ----------------------------------------------------------
    # 4. PREDIÇÃO ÚNICA
    # ----------------------------------------------------------
    def predict_single(self, x_query: np.ndarray, model_predictions: np.ndarray) -> float:
        """
        x_query           : (N_features,)
        model_predictions : (M_modelos,) — predições de todos os modelos para esta instância
        Retorna: média das predições dos modelos selecionados.
        """
        neighbor_idx = self._get_neighbors(x_query)
        local_errors = self._local_competence(neighbor_idx)
        n_select     = max(1, int(np.ceil(self.n_models * self.threshold)))
        selected     = np.argsort(local_errors)[:n_select]
        return float(np.mean(model_predictions[selected]))

    # ----------------------------------------------------------
    # 5. PREDIÇÃO EM LOTE
    # ----------------------------------------------------------
    def predict(self, X_test: np.ndarray, pool_predictions: np.ndarray) -> np.ndarray:
        """
        X_test           : (N_teste, N_features)
        pool_predictions : (N_teste, M_modelos)
        Retorna: array (N_teste,)
        """
        X_test           = np.array(X_test,           dtype=float)
        pool_predictions = np.array(pool_predictions, dtype=float)
        preds = np.array([
            self.predict_single(X_test[i], pool_predictions[i])
            for i in range(len(X_test))
        ])
        print(f'[DES] predict() | {len(preds)} instâncias | k={self.k} | top {self.threshold*100:.0f}% modelos')
        return preds

print('✅ T5 — DynamicEnsembleSelector carregado')

In [ ]:
# ── Teste isolado do T5 com mock data (Finnish: 283 treino / 122 teste) ──
np.random.seed(42)

N_TRAIN, N_TEST, N_FEAT, M_MODELS = 283, 122, 31, 11

print('\n[MOCK] Gerando dados fictícios para teste isolado do DES...')
X_train_mock  = np.random.rand(N_TRAIN, N_FEAT)
y_train_mock  = np.random.uniform(500, 60_000, N_TRAIN)
gabarito_mock = np.column_stack([
    y_train_mock * np.random.uniform(0.8, 1.2, N_TRAIN) for _ in range(M_MODELS)
])
X_test_mock   = np.random.rand(N_TEST, N_FEAT)
y_test_mock   = np.random.uniform(500, 60_000, N_TEST)
pool_test_mock = np.column_stack([
    y_test_mock * np.random.uniform(0.8, 1.2, N_TEST) for _ in range(M_MODELS)
])

des = DynamicEnsembleSelector(k=7, threshold=0.3)
des.fit(X_train_mock, gabarito_mock, y_train_mock)
y_pred_des = des.predict(X_test_mock, pool_test_mock)

evaluate_all(y_test_mock, y_pred_des, model_name='DES (Mock)')
print('\n[OK] T5 funcionando corretamente.')

---
## T7 — `statistical_tests.py` · Validação Estatística de Demšar

1. **Teste de Friedman** — compara N modelos em K datasets (não-paramétrico)  
2. **Post-hoc Bonferroni-Dunn** — compara cada modelo vs. o proposto **OES**  
3. **Critical Difference Diagram** — visualização estilo Demšar (2006)

In [ ]:
# ============================================================
#  T7 - statistical_tests.py
#  Responsável: A
# ============================================================

# ----------------------------------------------------------
# 1. TESTE DE FRIEDMAN
# ----------------------------------------------------------
def friedman_test(results_matrix: np.ndarray, model_names: list) -> dict:
    """
    results_matrix : (K_datasets, N_modelos) — valores de erro (menor = melhor)
    model_names    : lista de strings
    """
    K, N = results_matrix.shape
    assert len(model_names) == N, 'Número de nomes != número de colunas'

    ranks = np.zeros_like(results_matrix, dtype=float)
    for i in range(K):
        order = np.argsort(results_matrix[i])
        rank_row = np.empty_like(order, dtype=float)
        rank_row[order] = np.arange(1, N + 1)
        ranks[i] = rank_row

    avg_ranks    = np.mean(ranks, axis=0)
    sum_sq_ranks = np.sum(avg_ranks ** 2)
    F_friedman   = (12 * K / (N * (N + 1))) * (sum_sq_ranks - N * (N + 1) ** 2 / 4)
    p_value      = 1 - stats.chi2.cdf(F_friedman, df=N - 1)

    print('\n' + '='*55)
    print('  TESTE DE FRIEDMAN')
    print('='*55)
    print(f'  Estatística de Friedman : {F_friedman:.4f}')
    print(f'  p-value                 : {p_value:.6f}')
    print(f"  Significativo (p<0.05)  : {'✅ SIM' if p_value < 0.05 else '❌ NÃO'}")
    print('\n  Ranks médios por modelo:')
    for name, rank in sorted(zip(model_names, avg_ranks), key=lambda x: x[1]):
        bar = '█' * int(rank * 3)
        print(f'    {name:<20} rank={rank:.3f}  {bar}')
    print('='*55)

    return {
        'F_statistic' : F_friedman,
        'p_value'     : p_value,
        'avg_ranks'   : dict(zip(model_names, avg_ranks)),
        'rank_matrix' : ranks,
        'significant' : p_value < 0.05
    }


# ----------------------------------------------------------
# 2. POST-HOC: BONFERRONI-DUNN
# ----------------------------------------------------------
def bonferroni_dunn_test(friedman_result: dict, control_model: str,
                         n_datasets: int, alpha: float = 0.05) -> dict:
    """
    Compara cada modelo contra o modelo de controle (OES proposto).
    """
    avg_ranks    = friedman_result['avg_ranks']
    N            = len(avg_ranks)
    control_rank = avg_ranks[control_model]
    se           = np.sqrt(N * (N + 1) / (6 * n_datasets))
    n_comp       = N - 1
    z_alpha      = stats.norm.ppf(1 - alpha / (2 * n_comp))
    cd           = z_alpha * se

    print('\n' + '='*55)
    print(f'  POST-HOC: BONFERRONI-DUNN  (controle: {control_model})')
    print('='*55)
    print(f'  Diferença Crítica (CD) : {cd:.4f}')
    print(f'  z crítico (Bonferroni) : {z_alpha:.4f}')
    print(f'  alpha ajustado         : {alpha/n_comp:.5f}\n')

    results = {}
    for model, rank in avg_ranks.items():
        if model == control_model:
            continue
        diff        = abs(rank - control_rank)
        significant = diff > cd
        symbol      = ('✅ MELHOR' if (rank > control_rank and significant) else
                       '❌ PIOR'   if (rank < control_rank and significant) else '— empate')
        print(f'  {control_model} vs {model:<18}  |diff|={diff:.3f}  CD={cd:.3f}  {symbol}')
        results[model] = {'rank_diff': diff, 'significant': significant,
                          'cd': cd, 'control_better': rank > control_rank}
    print('='*55)
    return results


# ----------------------------------------------------------
# 3. CRITICAL DIFFERENCE DIAGRAM (Demšar 2006)
# ----------------------------------------------------------
def plot_cd_diagram(friedman_result: dict, control_model: str,
                    n_datasets: int, alpha: float = 0.05):
    """Gera e exibe o diagrama de Diferença Crítica no estilo Demšar (2006)."""
    avg_ranks = friedman_result['avg_ranks']
    N         = len(avg_ranks)
    se        = np.sqrt(N * (N + 1) / (6 * n_datasets))
    n_comp    = N - 1
    z_alpha   = stats.norm.ppf(1 - alpha / (2 * n_comp))
    cd        = z_alpha * se

    sorted_models = sorted(avg_ranks.items(), key=lambda x: x[1])
    names  = [m[0] for m in sorted_models]
    ranks  = [m[1] for m in sorted_models]

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.set_xlim(0.5, N + 0.5)
    ax.set_ylim(-1, 1)
    ax.axhline(0, color='black', lw=1.5)

    for i, (name, rank) in enumerate(zip(names, ranks)):
        color  = '#e74c3c' if name == control_model else '#2980b9'
        marker = '*' if name == control_model else 'o'
        ms     = 12 if name == control_model else 8
        ax.plot(rank, 0, marker, color=color, ms=ms, zorder=5)
        va     = 'bottom' if i % 2 == 0 else 'top'
        offset = 0.18 if i % 2 == 0 else -0.18
        ax.text(rank, offset, f'{name}\n({rank:.2f})',
                ha='center', va=va, fontsize=7.5, color=color, fontweight='bold')

    ctrl_rank = avg_ranks[control_model]
    ax.annotate('', xy=(ctrl_rank + cd, 0.6), xytext=(ctrl_rank, 0.6),
                arrowprops=dict(arrowstyle='<->', color='#27ae60', lw=2.5))
    ax.text(ctrl_rank + cd / 2, 0.72, f'CD = {cd:.2f}',
            ha='center', fontsize=9, color='#27ae60', fontweight='bold')

    ax.set_title(f'Critical Difference Diagram (Demšar 2006) — controle: {control_model}',
                 fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Rank médio  (menor = melhor)', fontsize=9)
    ax.set_yticks([])
    ax.spines[['left','right','top']].set_visible(False)
    plt.tight_layout()
    plt.show()
    print(f'[PLOT] Diagrama exibido | CD={cd:.4f}')

print('✅ T7 — statistical_tests carregado')

In [ ]:
# ── Teste isolado do T7 com mock data (Finnish + Maxwell) ──────
np.random.seed(42)

model_names = ['SVM','RF','MLP','kNN','DT','ET','LR',
               'ADA','CAT','XGB','NB','BG','Static','DES','OES']

# Tabela de sMAPE simulada (baseada nos valores do artigo)
results_mock = np.array([
    # Finnish
    [48.1, 29.1, 38.7, 35.6, 37.5, 27.7, 42.9,
     37.6, 33.5, 30.1, 38.7, 43.4, 27.8, 33.8, 23.9],
    # Maxwell
    [37.3, 18.2, 45.0, 35.0, 30.1, 18.5, 65.2,
     24.8, 36.4, 18.6, 39.1, 20.5, 23.5, 16.5, 15.0],
])

# 1. Friedman
fr_result = friedman_test(results_mock, model_names)

# 2. Bonferroni-Dunn
if fr_result['significant']:
    bd_result = bonferroni_dunn_test(fr_result, control_model='OES',
                                     n_datasets=2, alpha=0.05)
else:
    print('\n[INFO] Friedman não significativo — post-hoc não aplicável.')

# 3. CD Diagram
plot_cd_diagram(fr_result, control_model='OES', n_datasets=2, alpha=0.05)

print('\n[OK] T7 funcionando corretamente.')

---
## T8 — Integração com os dados reais de M

> Quando M entregar os arquivos reais, substitua os blocos abaixo.

In [ ]:
# ── PASSO 1: carregar dados reais de M ─────────────────────────
# Substitua pelo caminho correto após M subir no Drive/Git

# Opção A — Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# X_train = np.load('/content/drive/MyDrive/projeto_oes/X_train.npy')
# y_train = np.load('/content/drive/MyDrive/projeto_oes/y_train.npy')
# gabarito = np.load('/content/drive/MyDrive/projeto_oes/gabarito_treino.npy')
# X_test   = np.load('/content/drive/MyDrive/projeto_oes/X_test.npy')
# y_test   = np.load('/content/drive/MyDrive/projeto_oes/y_test.npy')
# pool_preds = np.load('/content/drive/MyDrive/projeto_oes/pool_predictions.npy')

# Opção B — upload direto no Colab
# from google.colab import files
# uploaded = files.upload()

print('⚠️  Célula de integração — aguardando dados reais de M.')
print('    Descomente o bloco correspondente quando os arquivos estiverem disponíveis.')

In [ ]:
# ── PASSO 2: rodar DES com dados reais ─────────────────────────
# des_real = DynamicEnsembleSelector(k=7, threshold=0.3)
# des_real.fit(X_train, gabarito, y_train)
# y_pred_real = des_real.predict(X_test, pool_preds)
# evaluate_all(y_test, y_pred_real, model_name='DES (Finnish real)')

print('⚠️  Descomente após carregar os dados reais no passo anterior.')

In [ ]:
# ── PASSO 3: testes estatísticos com resultados reais ──────────
# results_real = np.array([
#     [smape_svm_finnish, smape_rf_finnish, ..., smape_oes_finnish],
#     [smape_svm_maxwell, smape_rf_maxwell, ..., smape_oes_maxwell],
# ])
# fr_real = friedman_test(results_real, model_names)
# if fr_real['significant']:
#     bonferroni_dunn_test(fr_real, control_model='OES', n_datasets=2)
# plot_cd_diagram(fr_real, control_model='OES', n_datasets=2)

print('⚠️  Descomente após ter a tabela de resultados completa.')